# 🎙️ Unlimited Voice Cloning

In [1]:
%%bash
set -e

cd /content

# Download micromamba into /content/bin/micromamba
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create a clean env (isolated from Colab’s global packages)
./bin/micromamba create -y -n cb311 -c conda-forge python=3.11 pip

echo "✅ micromamba ready at /content/bin/micromamba"
echo "✅ env created: cb311"


bin/micromamba
Fetch Shard Index for conda-forge/linux-64                                                      ⧖ Starting
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.2 sec)
Fetch Shard Index for conda-forge/noarch                                                        ⧖ Starting
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.2 sec)
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (8.5 sec)

Resolving Environment                                                                           ⧖ Starting
Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /root/.local/share/mamba/envs/cb311

  Updating specs:

   - python=3.11
   - pip


  Package           

In [2]:
%%bash
set -euo pipefail

cd /content
MICROMAMBA="/content/bin/micromamba"

ts() { date +"[%Y-%m-%d %H:%M:%S]"; }

echo "$(ts) Sanity check micromamba:"
ls -lah "$MICROMAMBA"

echo "$(ts) Upgrading pip tooling inside cb311..."
"$MICROMAMBA" run -n cb311 python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CUDA 12.1)... (this can take a while)"
"$MICROMAMBA" run -n cb311 pip install \
  --progress-bar on \
  torch==2.5.1+cu121 torchaudio==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

echo "$(ts) Installing Chatterbox package (from GitHub, no-cache, upgrade)..."
"$MICROMAMBA" run -n cb311 pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n cb311 pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  "chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master"

echo "$(ts) Installing s3tokenizer + onnx (--no-deps to avoid protobuf conflict)..."
"$MICROMAMBA" run -n cb311 pip install --no-deps s3tokenizer==0.3.0 onnx==1.16.0

echo "$(ts) Force-upgrading protobuf for onnx compatibility..."
"$MICROMAMBA" run -n cb311 pip install --no-deps --force-reinstall "protobuf>=4.25.0"

echo "$(ts) ✅ Installation complete!"

[2026-06-23 07:04:01] Sanity check micromamba:
-rwxrwxr-x 1 root root 18M Jun  8 20:10 /content/bin/micromamba
[2026-06-23 07:04:01] Upgrading pip tooling inside cb311...
[2026-06-23 07:04:02] Installing PyTorch 2.5.1 (CUDA 12.1)... (this can take a while)
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 26.7 MB/s  0:00:14
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 125.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 102.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 135.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 160.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 50.7 MB/s  0:00:10
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 42.2 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/

  Running command git clone --filter=blob:none --quiet https://github.com/devnen/chatterbox-v2.git /tmp/pip-install-8vjetyi5/chatterbox-tts_ad9a02f5f14f4c4f9ccf6b9e60b4931a
  Running command git checkout -b master --track origin/master
  Switched to a new branch 'master'
  Branch 'master' set up to track remote branch 'master' from 'origin'.


In [3]:
%%bash
set -e

/content/bin/micromamba run -n cb311 python - <<'PY'
import inspect, torch
import chatterbox.tts_turbo as t

print("✅ torch:", torch.__version__)
print("✅ cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ gpu:", torch.cuda.get_device_name(0))

print("✅ chatterbox.tts_turbo path:", t.__file__)

src = inspect.getsource(t.ChatterboxTurboTTS.from_pretrained)
print("\n--- from_pretrained() (first ~80 lines) ---")
print("\n".join(src.splitlines()[:80]))

# Heuristic check for the common buggy pattern that forces token=True semantics
markers = [" or True", "token=True", "token = True", "use_auth_token=True"]
hits = [m for m in markers if m in src]
print("\nHeuristic auth-forcing markers found:", hits)

if hits:
    raise SystemExit(
        "\n❌ This install still appears to force HF auth.\n"
        "Re-run Cell 2 (it already uses --no-cache-dir --upgrade).\n"
    )

print("\n✅ Looks good: Turbo should download without requiring user tokens.")
PY

✅ torch: 2.5.1+cu121
✅ cuda available: True
✅ gpu: Tesla T4
✅ chatterbox.tts_turbo path: /root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages/chatterbox/tts_turbo.py

--- from_pretrained() (first ~80 lines) ---
    @classmethod
    def from_pretrained(cls, device) -> 'ChatterboxTurboTTS':
        # Check if MPS is available on macOS
        if device == "mps" and not torch.backends.mps.is_available():
            if not torch.backends.mps.is_built():
                print("MPS not available because the current PyTorch install was not built with MPS enabled.")
            else:
                print("MPS not available because the current MacOS version is not 12.3+ and/or you do not have an MPS-enabled device on this machine.")
            device = "cpu"

        local_path = snapshot_download(
            repo_id=REPO_ID,
            token=os.getenv("HF_TOKEN") or False,
            # Optional: Filter to download only what you need
            allow_patterns=["*.safetensors"

In [ ]:
import os, time, subprocess, socket, requests
from pathlib import Path

PORT = 8004
REPO_DIR = "/content/Unlimited-Voice-Cloning"
LOG_STDOUT = "/content/chatterbox_server_stdout.log"

def sh(cmd, check=False):
    return subprocess.run(["bash", "-lc", cmd], check=check)

def port_open(host="127.0.0.1", port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

os.chdir("/content")

# Fresh clone into the correct directory
sh(f"rm -rf {REPO_DIR}", check=False)
sh("git clone https://github.com/uzairaffliate-sketch/Unlimited-Voice-Cloning.git", check=True)
os.chdir(REPO_DIR)

print("=== Quick system checks ===")
sh("nvidia-smi || true", check=False)

print("\n=== Installing server requirements (prefer repo pins if present) ===")
if Path("requirements-nvidia.txt").exists():
    sh("/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel", check=False)
    sh("/content/bin/micromamba run -n cb311 pip install -r requirements-nvidia.txt", check=False)
else:
    sh(
        "/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel && "
        "/content/bin/micromamba run -n cb311 pip install "
        "fastapi 'uvicorn[standard]' pyyaml soundfile librosa safetensors "
        "python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm "
        "pydub audiotsm praat-parselmouth",
        check=False
    )

sh("/content/bin/micromamba run -n cb311 pip install --no-deps --force-reinstall 'protobuf>=4.25.0'", check=False)

print("\n=== Applying watermarker patch ===")
SITE_PKG = "/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages"
CB_DIR = Path(SITE_PKG) / "chatterbox"
SENTINEL = "# [patched: watermarker made optional]"
TARGET = "self.watermarker = perth.PerthImplicitWatermarker()"
patched = 0
for fname in ["tts.py", "tts_turbo.py", "mtl_tts.py", "vc.py"]:
    fp = CB_DIR / fname
    if not fp.exists():
        continue
    content = fp.read_text(encoding="utf-8")
    if SENTINEL in content or TARGET not in content:
        continue
    lines = content.split("\n")
    new_lines = []
    for line in lines:
        if TARGET in line and line.lstrip().startswith("self."):
            ind = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{ind}{SENTINEL}")
            new_lines.append(f"{ind}try:")
            new_lines.append(f"{ind}    self.watermarker = perth.PerthImplicitWatermarker()")
            new_lines.append(f"{ind}except Exception:")
            new_lines.append(f"{ind}    class _NoOpWatermarker:")
            new_lines.append(f"{ind}        def apply_watermark(self, wav, *args, **kwargs):")
            new_lines.append(f"{ind}            return wav")
            new_lines.append(f"{ind}    self.watermarker = _NoOpWatermarker()")
        else:
            new_lines.append(line)
    fp.write_text("\n".join(new_lines), encoding="utf-8")
    print(f"  Patched {fname}")
    patched += 1
if patched:
    print(f"  {patched} file(s) patched for optional watermarking")
else:
    print("  No patching needed")

print("\n=== Removing old stdout log ===")
Path(LOG_STDOUT).unlink(missing_ok=True)

print("\n=== Starting server with LIVE logs ===")
print("Log file:", LOG_STDOUT)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["HF_HOME"] = "/content/hf_home"
env["TRANSFORMERS_CACHE"] = "/content/hf_home/transformers"
env["HF_HUB_CACHE"] = "/content/hf_home/hub"
Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    ["/content/bin/micromamba", "run", "-n", "cb311", "python", "-u", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

with open(LOG_STDOUT, "w", encoding="utf-8", errors="replace") as f:
    shown_link = False
    while True:
        line = proc.stdout.readline()
        if line:
            print(line, end="")
            f.write(line)
            f.flush()

        if (not shown_link) and port_open():
            shown_link = True
            from google.colab.output import eval_js
            proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
            print(f"\n🌐 Open this URL: {proxy_url}\n")

        if proc.poll() is not None:
            print("\n=== Server process exited with code", proc.returncode, "===")
            break

=== Quick system checks ===

=== Installing server requirements (prefer repo pins if present) ===

=== Applying watermarker patch ===
  No patching needed

=== Removing old stdout log ===

=== Starting server with LIVE logs ===
Log file: /content/chatterbox_server_stdout.log
/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-06-23 07:26:59 [INFO] __main__: Starting TTS Server directly on http://0.0.0.0:8004
2026-06-23 07:26:59 [INFO] __main__: API documentation will be available at http://0.0.0.0:8004/docs
2026-06-23 07:26:59 [INFO] __main__: Web UI will be available at http://0.0.0.0:8004/
INFO:     Started server process [7988]
INFO:     Waiting for application startup.
2026-06-23 07:26:59 [INFO] server: TTS Server: Initializing application...
2026-06-23 07:26:59 [INFO] server: Configuration loaded